# AI Hiring Bias in the Age of Algorithmic Recruitment
## A Fairness Audit Using US Census Data

### The Case That Changed Everything

In 2023, Derek Mobley filed a lawsuit against Workday Inc. alleging that its
AI-powered applicant screening tool systematically rejected him and thousands
of other candidates based on age, race, and disability. He had applied to over
100 jobs through Workday's platform and was rejected every time — often within
minutes, sometimes in the middle of the night — before any human ever reviewed
his application.

In May 2025, a California federal court certified the case as a collective
action under the Age Discrimination in Employment Act, potentially covering
millions of applicants. Workday is not an edge case. 99% of Fortune 500
companies now use AI assistance in hiring decisions.

### The Regulatory Response

New York City's Local Law 144 (2023) now mandates independent bias audits
for any automated employment decision tool used in the city. The EU AI Act
(2024) classifies AI systems used in employment and recruitment as high-risk,
requiring documented bias assessments before deployment. The August 2026
compliance deadline is three months away.

Neither law tells organisations *how* to conduct a bias audit. That is the gap
fairpipe fills.

### What This Notebook Does

Using the American Community Survey (ACS) Public Use Microdata — the same
US Census data that underpins major labour market policy — we simulate the
type of bias audit that hiring algorithm developers and compliance teams now
need to conduct. We will:

1. **Measure** employment prediction bias by race and sex using rigorous
   statistical metrics with confidence intervals
2. **Detect** proxy variables and structural disparities in the feature set
3. **Mitigate** bias using fairpipe's pipeline and validate the improvement
4. **Show** how this entire workflow integrates into CI/CD via the
   fairpipe GitHub Action

This is not a theoretical exercise. This is exactly the analysis that Workday
should have conducted — and didn't — before deploying its screening tool at scale.

In [ ]:
# !pip install folktables --quiet
# !pip install fairpipe --quiet 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from folktables import ACSDataSource, ACSEmployment
from fairpipe import FairnessAnalyzer, load_data
from fairpipe.pipeline import load_config, build_pipeline, apply_pipeline, run_detectors
from fairpipe.integration import execute_workflow
 
THRESHOLD = 0.05
DECISION_THRESHOLD = 0.57
SURVEY_YEAR = "2018"
STATE = "CA"
 
# Race label mapping (ACS RAC1P codes)
RACE_LABELS = {
    1: "White",
    2: "Black",
    3: "American Indian",
    4: "Alaska Native",
    5: "Indigenous",
    6: "Asian",
    7: "Pacific Islander",
    8: "Other",
    9: "Two or More",
}
 
SEX_LABELS = {1: "Male", 2: "Female"}
 
# Load ACS 2018 California data — uses local cache, no download required
print("Loading ACS 2018 California data from cache...")
data_source = ACSDataSource(
    survey_year=SURVEY_YEAR,
    horizon="1-Year",
    survey="person",
)
acs_raw = data_source.get_data(states=[STATE], download=True)
print(f"Raw records: {len(acs_raw):,}")
 
# Build clean DataFrame
df = acs_raw[acs_raw["AGEP"] >= 16].copy()
df = df[["AGEP", "COW", "SCHL", "MAR", "OCCP", "POBP",
         "RELP", "WKHP", "SEX", "RAC1P", "ESR"]].dropna().copy()
 
df["employed"]   = (df["ESR"] == 1.0).astype(int)
df["race_label"] = df["RAC1P"].map(RACE_LABELS)
df["sex_label"]  = df["SEX"].map(SEX_LABELS)
 
print(f"\nAnalysis dataset: {len(df):,} individuals (age 16+)")
print(f"Overall employment rate: {df['employed'].mean():.1%}")
print(f"\nDataset: ACS {SURVEY_YEAR} · State: {STATE} · Task: Employment Prediction")

In [ ]:
print("=== Employment Rates by Race ===")
race_stats = df.groupby("race_label")["employed"].agg(
    Count="count",
    Employment_Rate="mean"
).sort_values("Count", ascending=False)
race_stats["Employment_Rate"] = race_stats["Employment_Rate"].map("{:.1%}".format)
print(race_stats.to_string())
 
print("\n=== Employment Rates by Sex ===")
sex_stats = df.groupby("sex_label")["employed"].agg(
    Count="count",
    Employment_Rate="mean"
)
sex_stats["Employment_Rate"] = sex_stats["Employment_Rate"].map("{:.1%}".format)
print(sex_stats.to_string())
 
print("\n=== Employment Rates by Race × Sex (White, Black, Asian) ===")
intersect_stats = df[df["race_label"].isin(["White", "Black", "Asian"])].groupby(
    ["race_label", "sex_label"]
)["employed"].agg(
    Count="count",
    Employment_Rate="mean"
)
intersect_stats["Employment_Rate"] = intersect_stats["Employment_Rate"].map("{:.1%}".format)
print(intersect_stats.to_string())

### What the Data Tells Us Before Any Model Runs

The 2018 ACS California dataset covers 196,604 working-age individuals and
reveals employment disparities that exist before any algorithm is involved.
Understanding the baseline is essential — it tells us how much of what a
model learns reflects genuine labour market differences, and how much reflects
structural inequality baked into the training data.

Three findings stand out:

**The Black–White employment gap is 4.4 percentage points.** Black Californians
are employed at 83.3% compared to 87.7% for white Californians. This gap is
not explained by qualifications alone — it reflects decades of documented
structural barriers in housing, education, and hiring. When an algorithm trains
on this data without fairness constraints, it learns to replicate this gap.

**The intersectional gap for Black men is 7.0 percentage points.** Black men
are employed at 81.0% compared to 88.0% for white men. This is the same
demographic at the centre of the Workday lawsuit — algorithmically screened
out at a measurably higher rate. The raw data already contains this signal
before a single model is trained.

**The gender gap is smaller but legally significant.** A 0.8 percentage point
overall gap between men and women grows when examined by race: Black women
face a 1.8 point gap versus white women, and Black men face a 7.0 point gap
versus white men. This intersectional pattern is precisely what NYC Local Law
144 and EU AI Act bias audits are designed to surface.

For the fairness analysis that follows, we focus on the **White–Black
comparison** — the largest racial gap, the best-powered statistically, and
the most directly relevant to active AI hiring litigation.

###  Train a Hiring Prediction Model

In [ ]:
# Features — exclude sensitive attributes (RAC1P, SEX) and proxies (WKHP)
FEATURES = ["AGEP", "SCHL", "MAR", "POBP", "RELP"]
 
# Focus on White and Black Californians
df_wb = df[df["race_label"].isin(["White", "Black"])].copy()
df_wb = df_wb[FEATURES + ["employed", "race_label", "sex_label"]].dropna().copy()
 
print(f"Analysis subset: {len(df_wb):,} individuals")
print(f"White: {(df_wb['race_label']=='White').sum():,}")
print(f"Black: {(df_wb['race_label']=='Black').sum():,}")
print(f"\nFeatures: {FEATURES}")
print("(RAC1P, SEX, WKHP excluded — sensitive attributes and target proxies)")
 
X = df_wb[FEATURES]
y = df_wb["employed"]
 
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df_wb.index,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
 
# class_weight="balanced" prevents the model from predicting everyone
# as employed to achieve cheap accuracy on this imbalanced dataset
clf = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced",
)
clf.fit(X_train_scaled, y_train)
 
# DECISION_THRESHOLD=0.57 simulates a selective hiring screener that
# passes roughly the top third of candidates — realistic for AI tools
# used to thin large applicant pools before human review
y_prob = clf.predict_proba(X_test_scaled)[:, 1]
 
df_test = df_wb.loc[idx_test].copy()
df_test["y_score"] = y_prob
df_test["y_pred"]  = (y_prob >= DECISION_THRESHOLD).astype(int)
df_test["y_true"]  = y_test.values
 
accuracy = (df_test["y_pred"] == df_test["y_true"]).mean()
print(f"\nModel accuracy:         {accuracy:.1%}")
print(f"Decision threshold:     {DECISION_THRESHOLD}")
print(f"Overall selection rate: {df_test['y_pred'].mean():.1%}")
print(f"Test set size:          {len(df_test):,}")
 
print("\n=== Selection Rates by Race ===")
print(df_test.groupby("race_label")["y_pred"].mean().round(4).to_string())
 
print("\n=== Predicted vs Actual Employment Rates ===")
pred_rates = df_test.groupby("race_label").agg(
    Count=("y_pred", "count"),
    Actual_Rate=("y_true", "mean"),
    Selection_Rate=("y_pred", "mean"),
)
print(pred_rates.round(4).to_string())
 

### Measure Fairness

In [ ]:
analyzer = FairnessAnalyzer.from_dataframe(
    df_test,
    y_pred_col="y_pred",
    sensitive_col="race_label",
    y_true_col="y_true",
    min_group_size=30,
)
 
dpd = analyzer.demographic_parity_difference(with_ci=True)
eod = analyzer.equalized_odds_difference(with_ci=True)
 
print("=== Fairness Metrics — Hiring Model (White vs Black) ===")
print()
print(f"Demographic Parity Difference:  {dpd.value:.4f}")
print(f"95% CI:                         [{dpd.ci[0]:.4f}, {dpd.ci[1]:.4f}]")
print()
print(f"Equalized Odds Difference:      {eod.value:.4f}")
print(f"95% CI:                         [{eod.ci[0]:.4f}, {eod.ci[1]:.4f}]")
print()
print(f"DPD vs threshold ({THRESHOLD}): "
      f"{'✅ PASSED' if dpd.value <= THRESHOLD else '❌ FAILED'}")
print(f"EOD vs threshold ({THRESHOLD}): "
      f"{'✅ PASSED' if eod.value <= THRESHOLD else '❌ FAILED'}")
print()
 
print("=== Error Rates by Race ===")
for race in ["White", "Black"]:
    g   = df_test[df_test["race_label"] == race]
    fpr = ((g["y_pred"]==1) & (g["y_true"]==0)).sum() / max((g["y_true"]==0).sum(), 1)
    fnr = ((g["y_pred"]==0) & (g["y_true"]==1)).sum() / max((g["y_true"]==1).sum(), 1)
    print(f"{race:<8}  n={len(g):,}  FPR={fpr:.3f}  FNR={fnr:.3f}")
 

### Step 1: Measuring the Bias

The model selects white candidates at **32.3%** and Black candidates at
**21.8%** — a gap of **10.5 percentage points**. Both fairness metrics
confirm this disparity is statistically significant and practically meaningful.

**Demographic Parity Difference: 0.1046**
(95% CI: [0.0852, 0.1240])

A Black candidate is 10.5 percentage points less likely to be selected by
this algorithm than a white candidate with the same features. The confidence
interval sits entirely above zero — this is not noise. The lower bound of
0.0852 alone is nearly twice the 0.05 regulatory threshold.

**Equalized Odds Difference: 0.1022**
(95% CI: [0.0847, 0.1361])

The false negative rate tells the deeper story. The model incorrectly
rejects **76.8%** of Black candidates who are actually employed, compared
to **66.6%** of white candidates in the same situation. In a hiring context
a false negative means a qualified candidate is screened out before a human
ever sees their application.

In the Workday lawsuit, plaintiffs alleged they were rejected within minutes
of applying — often in the middle of the night — before any human review.
This is what that looks like in the data: a 10-point gap in selection rates
between equally qualified candidates of different races, generated entirely
by a model that never saw the word "race" in its feature set.

**Both metrics fail the 0.05 threshold.** Under NYC Local Law 144, this
model could not be deployed in New York City without remediation. Under the
EU AI Act, deploying it in a high-risk employment context without documented
bias assessment and mitigation would expose the company to fines of up to
€35 million.

The algorithm did not intend to discriminate. It never does.
It learned from data that already carried the weight of structural inequality
— in education access, in occupational segregation, in geographic
distribution — and faithfully reproduced that inequality at scale.
This is why measurement is not optional.

### Visualize the Disparity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
 
# Chart 1 — Selection rates
races  = ["White", "Black"]
rates  = [df_test[df_test["race_label"]==r]["y_pred"].mean() for r in races]
colors = ["#1f77b4", "#d62728"]
axes[0].bar(races, rates, color=colors)
axes[0].axhline(THRESHOLD*6, color="grey", linestyle="--", alpha=0.5)
axes[0].set_title("Selection Rate by Race\n(threshold = 0.57)")
axes[0].set_ylabel("Selection Rate")
axes[0].set_ylim(0, 0.5)
for i, v in enumerate(rates):
    axes[0].text(i, v + 0.005, f"{v:.1%}", ha="center", fontsize=11)
 
# Chart 2 — False negative rates
fnrs = []
for race in races:
    g   = df_test[df_test["race_label"]==race]
    fnr = ((g["y_pred"]==0) & (g["y_true"]==1)).sum() / max((g["y_true"]==1).sum(), 1)
    fnrs.append(fnr)
 
axes[1].bar(races, fnrs, color=colors)
axes[1].set_title("False Negative Rate by Race\n(qualified candidates rejected)")
axes[1].set_ylabel("False Negative Rate")
axes[1].set_ylim(0, 1.0)
for i, v in enumerate(fnrs):
    axes[1].text(i, v + 0.01, f"{v:.1%}", ha="center", fontsize=11)
 
plt.suptitle("ACS Employment Model — Racial Bias in Hiring Predictions",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("acs_bias_charts.png", dpi=150, bbox_inches="tight")
plt.show()
 

## Bias Detection

### Step 2: Detecting and Mitigating the Bias
 
Before intervening, we run fairpipe's bias detectors — a suite of statistical
tests that scan the full feature set for representation imbalance, proxy
variables, and distributional disparities across sensitive groups.
 
This step answers a critical question: if we remove the `race_label` column,
does the model still have access to racial information through correlated
features? If so, removing race alone is insufficient — the model will learn
to discriminate through proxies.
"""

In [ ]:
config_yaml = f"""
sensitive: ["race_label"]
features: {FEATURES}
pipeline:
  - name: reweigh
    transformer: "InstanceReweighting"
training:
  method: "reductions"
  target_column: "employed"
  params:
    constraint: "demographic_parity"
    eps: 0.05
fairness_metric: "equalized_odds_difference"
validation_threshold: 0.05
"""
 
with open("acs_config.yml", "w") as f:
    f.write(config_yaml)
 
config = load_config("acs_config.yml")
 
detector_report = run_detectors(df=df_wb, cfg=config)
summary = detector_report.body["summary"]
 
print("=== Bias Detection Summary ===")
print(f"Sensitive attribute:   race_label")
print(f"Disparity flags:       {summary['disparity_flags']}  "
      f"(features with statistically significant racial disparity)")
print(f"Proxy flags:           {summary['proxy_flags']}  "
      f"(features that could act as proxies for race)")
print(f"Representation flags:  {summary['representation_flags']}  "
      f"(group size imbalances)")
 
proxies = detector_report.body["proxies"]
strong_proxies = sorted(
    [p for p in proxies if p["flagged"]],
    key=lambda x: x["strength"],
    reverse=True
)[:5]
 
if strong_proxies:
    print("\n--- Top Flagged Proxy Features ---")
    for p in strong_proxies:
        print(f"  {p['feature']:<20}  strength = {p['strength']:.3f}")
else:
    print("\nNo proxy features flagged above threshold.")
 
# Flag all disparity features for reference
print("\n--- Features with Statistically Significant Racial Disparity ---")
disparities = [
    d for d in detector_report.body["disparities"] if d["flagged"]
]
for d in disparities:
    print(f"  {d['feature']:<20}  p = {d['pvalue']:.2e}")

### What the Detector Found

The detection results tell a more complete story than a simple proxy flag
would suggest.

**No proxy features were flagged above the Cramér's V threshold of 0.3.**
At first glance this looks reassuring — none of the five features are
strong individual proxies for race. But look more carefully at the disparity
flags.

**Every single feature in the model shows a statistically significant
racial disparity.** All five features used for prediction — age, education,
marital status, place of birth, and relationship status — are distributed
differently across racial groups. The p-values are not marginal:

- Marital status (MAR): p = 1.37×10⁻²⁴¹
- Relationship status (RELP): p = 1.41×10⁻¹⁹³
- Age (AGEP): p = 1.71×10⁻¹⁴
- Place of birth (POBP): p = 2.34×10⁻¹⁴
- Education (SCHL): p = 1.12×10⁻²

These are not borderline results. They are among the most statistically
certain findings possible in a dataset of this size.

**The target variable itself is racially disparate.** The `employed` label
shows p = 2.69×10⁻³². The model is being trained to predict an outcome
that is already distributed unequally by race — which means a model that
perfectly predicts the training labels will by definition reproduce racial
disparities in its outputs.

**This is proxy discrimination in its most pervasive form.** There are no
strong individual proxies to drop because the racial signal is not
concentrated in one or two features — it is distributed across all of them.
Removing any single feature would not meaningfully reduce the disparity.
The only intervention that addresses this pattern is reweighting the
training distribution, which is what Instance Reweighting does.

This is precisely why the EU AI Act requires bias assessment across the
full feature set, not just a check for whether protected attributes appear
in the input. The absence of a race column does not equal the absence of
racial bias.

## Mitigation and Validation

### Step 3: Mitigation and Validation
 
With the bias quantified and the feature-level disparities mapped, we now
apply Instance Reweighting and retrain. fairpipe's `execute_workflow` runs
the complete three-step process in a single call:
 
1. **Baseline measurement** — trains a logistic regression on the specified
   features and measures fairness on the held-out test set
2. **Reweighting** — applies Instance Reweighting and retrains with the
   corrected sample weights
3. **Validation** — compares before/after metrics against the threshold
 
The `class_weight` and `decision_threshold` parameters added in v0.9.1
ensure the internal model matches the baseline model we trained above —
same class weighting, same selection threshold, valid before/after
comparison.


In [ ]:
result = execute_workflow(
    config=config,
    df=df_wb,
    output_dir=None,
    min_group_size=30,
    train_size=0.8,
    class_weight="balanced",
    decision_threshold=DECISION_THRESHOLD,
)
 
vr = result.validation_result
improvement_pct = (abs(vr.improvement) / vr.baseline_metric_value) * 100
 
print("=== Before vs After Mitigation (execute_workflow) ===")
print(f"{'Metric':<6}  {'Before':>8}  {'After':>8}  "
      f"{'Change':>8}  {'Threshold':>10}  {'Status'}")
print(f"{'-'*62}")
print(f"{'EOD':<6}  {vr.baseline_metric_value:>8.4f}  "
      f"{vr.final_metric_value:>8.4f}  "
      f"{vr.final_metric_value - vr.baseline_metric_value:>+8.4f}  "
      f"{THRESHOLD:>10.2f}  "
      f"{'✅ PASSED' if vr.passed else '❌ FAILED'}")
print()
print(f"Improvement:                {improvement_pct:.1f}%")
print(f"Remaining gap to threshold: "
      f"{max(0, vr.final_metric_value - THRESHOLD):.4f}")
print()
print(f"Pipeline status BEFORE: ❌ FAILED")
print(f"Pipeline status AFTER:  {'✅ PASSED' if vr.passed else '❌ FAILED'}")
print(f"\nMessage: {vr.message}")

### What This Tells Us (Ignore the Numpy Warning)

`execute_workflow` ran the complete pipeline with the same class weighting
and decision threshold as the baseline, producing a valid before/after
comparison. The results:

- EOD dropped from **0.1022 to 0.0536** — a reduction of **0.0487 (47.6%)**
- The remaining gap to the 0.05 threshold is just **0.0036**
- The pipeline correctly reports **❌ FAILED** — the model came within
  0.0036 of passing, but fairpipe enforces the threshold strictly

A 47.6% reduction in Equalized Odds Difference is a meaningful result for
a single pre-processing intervention on five features. The fair model is
now selecting Black and white candidates at rates that are 5.4 percentage
points apart rather than 10.5 — a measurable improvement in the
qualification of candidates who make it through to human review.

The remaining 0.0036 gap is narrow. Closing it would likely require one
or more of:

- **Stronger reweighting** — increasing the reweighting strength parameter
  in the config
- **Constraint-based training** — using `ReductionsWrapper` with an
  explicit equalized odds constraint rather than a demographic parity
  approximation
- **Feature engineering** — creating less racially disparate representations
  of age, education, and place of birth

The pipeline correctly flags this as FAILED. That strictness is the point.
A tool that rounds 0.0536 to "close enough" is not a compliance tool — it
is a liability. fairpipe leaves the threshold decision to the practitioner
and enforces it without exception.

## Conclusions

This notebook demonstrated a complete AI hiring bias audit on 2018 US
Census data, using **fairpipe** to measure, detect, mitigate, and validate
racial disparities in employment prediction. The key results:

- The ACS 2018 California dataset shows a **4.4 percentage point
  Black–White employment gap** before any model is trained — structural
  inequality encoded in the training data itself
- A logistic regression hiring screener selected white candidates at
  **32.3%** and Black candidates at **21.8%** — a gap of **10.5 percentage
  points** — without ever seeing a race feature
- Demographic Parity Difference: **0.1046** (95% CI: [0.0852, 0.1240])
- Equalized Odds Difference: **0.1022** (95% CI: [0.0847, 0.1361])
- Both confidence intervals sit entirely above zero — this is not noise
- Both metrics fail the 0.05 threshold — this model could not be deployed
  under NYC Local Law 144 without remediation
- The bias detector identified **7 features with statistically significant
  racial disparities** — every single feature used for prediction is
  racially disparate, meaning removing the race column alone would not
  fix this model
- `execute_workflow` reduced EOD from **0.1022 to 0.0536** — a **47.6%
  improvement** — bringing the model to within **0.0036** of the 0.05
  fairness threshold

This is the analysis teams building hiring tools are now legally required
to conduct. NYC Local Law 144 mandates it. The EU AI Act enforces it with
fines of up to €35 million. fairpipe makes it reproducible, automatable,
and CI/CD-ready in a single config file and a single function call.

The Workday lawsuit alleged that millions of applicants were screened out
by an algorithm before any human reviewed their application. This notebook
shows exactly how that happens — and exactly how to measure, quantify, and
begin to remediate it.

---

### Try It Yourself

[![Launch in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/SvrusIO/fAIr/main?urlpath=%2Fdoc%2Ftree%2Fcase_studies%2Facs_employment.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SvrusIO/fAIr/blob/main/case_studies/acs_employment.ipynb)

```bash
pip install fairpipe
```

**→ [GitHub](https://github.com/SvrusIO/fAIr) · [PyPI](https://pypi.org/project/fairpipe/) · [GitHub Action](https://github.com/SvrusIO/fairpipe-action)**

*Built by [Svrus](https://github.com/SvrusIO) — responsible intelligence in action*